# SwaggerLM — Fine-tune Qwen2.5-Coder-3B
**Before running:** Runtime → Change runtime type → **T4 GPU**

| | |
|---|---|
| Model | `unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit` |
| Task | Python FastAPI endpoint → OpenAPI JSON |
| Method | QLoRA rank 16 |
| Dataset | 8,074 train / 425 val |
| Sources | APIs.guru (656 providers) + FastAPI repos |

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────
%%capture
!pip install unsloth wandb
!pip install --upgrade bitsandbytes peft trl transformers accelerate datasets
print('✅ Done')

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), '❌ No GPU — Runtime → Change runtime type → T4'
gpu = torch.cuda.get_device_properties(0)
print(f'✅ GPU  : {gpu.name}')
print(f'✅ VRAM : {gpu.total_memory / 1024**3:.1f} GB')
print(f'✅ CUDA : {torch.version.cuda}')

In [ ]:
# ── Cell 3: Mount Drive & copy files ─────────────────────────────────
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil

DRIVE_DIR = '/content/drive/MyDrive/SwaggerLM'
os.makedirs(DRIVE_DIR, exist_ok=True)

for fname in ('train.jsonl', 'val.jsonl'):
    src = f'{DRIVE_DIR}/{fname}'
    if not os.path.exists(src):
        raise FileNotFoundError(
            f'Not found: {src}\n'
            f'Upload train.jsonl and val.jsonl to Google Drive → SwaggerLM/'
        )
    shutil.copy(src, fname)
    print(f'✅ {fname}  ({os.path.getsize(fname)/1024**2:.1f} MB)')

In [ ]:
# ── Cell 4: Config & load dataset ────────────────────────────────────
import json, random
from datasets import Dataset

# ════════════════════════════════════════
SMOKE_TEST_ROWS = 1_000   # ← set to None for full run
MAX_SEQ_LENGTH  = 1024
SEED            = 42
# ════════════════════════════════════════

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = load_jsonl('train.jsonl')
val_raw   = load_jsonl('val.jsonl')

if SMOKE_TEST_ROWS:
    rng = random.Random(SEED)
    train_raw = rng.sample(train_raw, min(SMOKE_TEST_ROWS, len(train_raw)))
    val_raw   = rng.sample(val_raw,   min(SMOKE_TEST_ROWS // 5, len(val_raw)))

print(f'Train : {len(train_raw):,} rows')
print(f'Val   : {len(val_raw):,} rows')

In [ ]:
# ── Cell 5: Format into Qwen2.5 chat template ─────────────────────────
SYSTEM = (
    'You are an expert API developer. '
    'Given a Python FastAPI endpoint, generate a complete and valid '
    'OpenAPI JSON documentation object. '
    'Include summary, description, parameters, requestBody if applicable, '
    'and responses with status codes. '
    'Output ONLY the JSON object, nothing else.'
)

def fmt(rec):
    return {'text': (
        f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
        f'<|im_start|>user\n'
        f'{rec["instruction"]}\n\n'
        f'```python\n{rec["input"]}\n```'
        f'<|im_end|>\n'
        f'<|im_start|>assistant\n'
        f'{rec["output"]}'
        f'<|im_end|>'
    )}

train_ds = Dataset.from_list([fmt(r) for r in train_raw])
val_ds   = Dataset.from_list([fmt(r) for r in val_raw])
print(f'✅ Formatted — train: {len(train_ds):,}  val: {len(val_ds):,}')
print('\n--- Sample prompt (truncated) ---')
print(train_ds[0]['text'][:600])

In [ ]:
# ── Cell 6: Load model ───────────────────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
print('✅ Qwen2.5-Coder-3B loaded')

In [ ]:
# ── Cell 7: Attach LoRA ──────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ['q_proj','k_proj','v_proj','o_proj',
                                   'gate_proj','up_proj','down_proj'],
    lora_alpha                 = 16,
    lora_dropout               = 0.0,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = SEED,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA attached — {trainable:,} trainable params ({100*trainable/total:.2f}%)')

In [ ]:
# ── Cell 7b: W&B init ────────────────────────────────────────────────
!pip install wandb -q
import wandb
wandb.login()  # paste your API key from wandb.ai/settings

wandb.init(
    project = 'swaggerlm',
    name    = 'qwen2.5-coder-3b-smoke-test' if SMOKE_TEST_ROWS else 'qwen2.5-coder-3b-full',
    config  = {
        'model':        'Qwen2.5-Coder-3B-Instruct',
        'lora_rank':    16,
        'dataset_size': len(train_ds),
        'seq_length':   MAX_SEQ_LENGTH,
        'epochs':       1,
        'batch_size':   16,
        'smoke_test':   bool(SMOKE_TEST_ROWS),
    }
)
print('✅ W&B initialized')

In [ ]:
# ── Cell 8: Trainer ──────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = True,
    args = SFTConfig(
        max_length                  = None,
        padding_free                = False,
        num_train_epochs            = 1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps                = 20,
        learning_rate               = 5e-5,
        lr_scheduler_type           = 'cosine',
        fp16 = not is_bfloat16_supported(),
        bf16 =     is_bfloat16_supported(),
        eval_strategy               = 'steps',
        eval_steps                  = 25,
        logging_steps               = 1,
        logging_strategy            = 'steps',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'eval_loss',
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        output_dir                  = './checkpoints',
        save_strategy               = 'steps',
        save_steps                  = 50,
        save_total_limit            = 2,
        seed                        = SEED,
        report_to                   = 'wandb',
    ),
)
print('✅ Trainer configured')

In [ ]:
# ── Cell 9: Drive backup callback ────────────────────────────────────
from transformers import TrainerCallback
import shutil, os

DRIVE_CKPT = '/content/drive/MyDrive/SwaggerLM/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

class DriveBackup(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        src = f'{args.output_dir}/checkpoint-{state.global_step}'
        dst = f'{DRIVE_CKPT}/checkpoint-{state.global_step}'
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'  ☁️  checkpoint-{state.global_step} → Drive')

trainer.add_callback(DriveBackup())
print('✅ Drive backup attached')

In [ ]:
# ── Cell 10: Train ───────────────────────────────────────────────────
import time, torch, gc

gc.collect()
torch.cuda.empty_cache()
used  = torch.cuda.memory_reserved(0) / 1024**3
total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'VRAM before training: {used:.1f} / {total_vram:.1f} GB')

def latest_checkpoint(drive_ckpt_dir):
    ckpts = sorted(
        [d for d in os.listdir(drive_ckpt_dir) if d.startswith('checkpoint-')],
        key=lambda x: int(x.split('-')[1])
    )
    return f'{drive_ckpt_dir}/{ckpts[-1]}' if ckpts else None

RESUME = None
# RESUME = latest_checkpoint(DRIVE_CKPT)  # ← uncomment after crash

t0 = time.time()
trainer.train(resume_from_checkpoint=RESUME)
print(f'\n⏱  Done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Cell 11: Loss curve ──────────────────────────────────────────────
import matplotlib.pyplot as plt

log      = trainer.state.log_history
t_steps  = [x['step']      for x in log if 'loss'      in x]
t_losses = [x['loss']      for x in log if 'loss'      in x]
v_steps  = [x['step']      for x in log if 'eval_loss' in x]
v_losses = [x['eval_loss'] for x in log if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_steps, t_losses, label='Train loss',      color='#2563eb', linewidth=2)
ax.plot(v_steps, v_losses, label='Validation loss', color='#dc2626', linewidth=2, linestyle='--')
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.set_title('SwaggerLM — Qwen2.5-Coder-3B Loss Curve', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
plt.savefig('loss_curve_swagger.png', dpi=150)
shutil.copy('loss_curve_swagger.png',
            '/content/drive/MyDrive/SwaggerLM/loss_curve.png')
plt.show()
print('✅ Saved locally + to Drive')

In [ ]:
# ── Cell 12: Inference test ──────────────────────────────────────────
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

TEST = '''
@router.get("/users/{user_id}")
async def get_user(
    user_id: int,
    include_posts: Optional[bool] = False,
    db: Session = Depends(get_db)
):
    """Get a user by their ID."""
    return db.query(User).filter(User.id == user_id).first()
'''.strip()

prompt = (
    f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
    f'<|im_start|>user\n'
    f'Generate a complete OpenAPI JSON documentation object for this FastAPI endpoint.\n\n'
    f'```python\n{TEST}\n```'
    f'<|im_end|>\n'
    f'<|im_start|>assistant\n'
)

inputs  = tokenizer(prompt, return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens     = 400,
    temperature        = 0.1,
    do_sample          = True,
    repetition_penalty = 1.1,
)
response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)
print('📝 Input endpoint:')
print(TEST)
print('\n🤖 Generated OpenAPI JSON:')
print(response)

# validate JSON
import json
try:
    parsed = json.loads(response)
    print('\n✅ Valid JSON!')
except json.JSONDecodeError as e:
    print(f'\n⚠️  Invalid JSON: {e}')

In [ ]:
# ── Cell 13: Save adapter to Drive ───────────────────────────────────
ADAPTER = 'swaggerlm_adapter'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)

drive_adapter = f'/content/drive/MyDrive/SwaggerLM/{ADAPTER}'
shutil.copytree(ADAPTER, drive_adapter, dirs_exist_ok=True)
wandb.finish()
print(f'✅ Adapter saved → Drive/SwaggerLM/{ADAPTER}/')